In [37]:
import pandas as pd
import numpy as np
import warnings
# 1. 데이터 불러오기
df = pd.read_csv("../Data/data_linier.csv")
df.head() 

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


## 알고리즘 판단
- **타깃(MEDV)**: 주택 중앙값(연속값) → **Linear Regression** 사용
- Logistic Regression은 분류(이진/다중 클래스)에 사용됨 

In [38]:

print("결측치 before:", df.isnull().sum().sum())
def clean_missing_values(df):
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == 'object':
                df[col] = df[col].fillna(df[col].mode()[0])
            else:
                df[col] = df[col].fillna(df[col].median())
    return df
df = clean_missing_values(df)
print("결측치 after:", df.isnull().sum().sum())
df.info()

결측치 before: 5
결측치 after: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 511 entries, 0 to 510
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     511 non-null    float64
 1   ZN       511 non-null    float64
 2   INDUS    511 non-null    float64
 3   CHAS     511 non-null    int64  
 4   NOX      511 non-null    float64
 5   RM       511 non-null    float64
 6   AGE      511 non-null    float64
 7   DIS      511 non-null    float64
 8   RAD      511 non-null    int64  
 9   TAX      511 non-null    int64  
 10  PTRATIO  511 non-null    float64
 11  B        511 non-null    float64
 12  LSTAT    511 non-null    float64
 13  MEDV     511 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 56.0 KB


In [39]:
# 3. Encoding (범주형 컬럼이 있으면 적용 - 이 데이터는 모두 수치형)
obj_cols = df.select_dtypes(include=['object']).columns
if len(obj_cols) > 0:
    for col in obj_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    print("Encoding 완료:", obj_cols.tolist())
else:
    print("범주형 컬럼 없음 - Encoding 스킵 (모두 수치형)")


범주형 컬럼 없음 - Encoding 스킵 (모두 수치형)


In [40]:
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [41]:
from sklearn.linear_model import LinearRegression
# 4. Scaling
scaler = StandardScaler()
feature_cols = [c for c in df.columns if c != 'MEDV']
X = df[feature_cols]
y = df['MEDV']
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
print("Scaling 완료 (StandardScaler)")
X_scaled.head()

Scaling 완료 (StandardScaler)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,-0.418162,0.290691,-1.296065,-0.271163,-0.145462,0.411862,-0.121697,0.146011,-0.977576,-0.664367,-1.455741,0.443853,-1.014091
1,-0.415709,-0.484767,-0.598270,-0.271163,-0.744435,0.191768,0.366340,0.564360,-0.862368,-0.986295,-0.318443,0.443853,-0.480058
2,-0.415712,-0.484767,-0.598270,-0.271163,-0.744435,1.283664,-0.267752,0.564360,-0.862368,-0.986295,-0.318443,0.399027,-1.136046
3,-0.415118,-0.484767,-1.315123,-0.271163,-0.839924,1.016407,-0.812787,1.086688,-0.747160,-1.105528,0.090984,0.418852,-1.275973
4,-0.410831,-0.484767,-1.315123,-0.271163,-0.839924,1.229355,-0.513552,1.086688,-0.747160,-1.105528,0.090984,0.443853,-0.969161


In [42]:
from sklearn.model_selection import train_test_split
# 5. Train/Test split & Linear Regression 학습
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
print("✅ Linear Regression 학습 완료")
print(f"Train R²: {model.score(X_train, y_train):.4f}")
print(f"Test R²:  {model.score(X_test, y_test):.4f}")

✅ Linear Regression 학습 완료
Train R²: 0.6969
Test R²:  0.1694


In [43]:
# 6. Predict 결과
y_pred = model.predict(X_test)
result_df = pd.DataFrame({
    '실제값(y_test)': y_test.values,
    '예측값(y_pred)': y_pred,
    '오차': np.abs(y_test.values - y_pred)
})
result_df.index = y_test.index
print("📌 Predict 결과 (테스트셋):")
result_df

📌 Predict 결과 (테스트셋):


,실제값(y_test),예측값(y_pred),오차
124,18.8,21.399688,2.599688
84,23.9,25.269267,1.369267
433,14.3,17.406818,3.106818
255,20.9,21.410889,0.510889
68,17.4,17.559463,0.159463
...,...,...,...
494,24.5,21.637151,2.862849
483,21.8,19.916162,1.883838
275,32.0,33.791807,1.791807
454,14.9,16.203778,1.303778


In [44]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# R² Score
r2 = r2_score(y_test, y_pred)
print(f"R2 Score:  {r2:.4f}")
# MSE (Mean Squared Error)
mse = mean_squared_error(y_test, y_pred)
print(f"MSE:      {mse:.4f}")
# RMSE (Root Mean Squared Error)
rmse = np.sqrt(mse)
print(f"RMSE:     {rmse:.4f}")
# MAE (Mean Absolute Error)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:      {mae:.4f}")

R2 Score:  0.1694
MSE:      71.1878
RMSE:     8.4373
MAE:      4.1437
